# Computation of Vela's Light Curve

## imports

In [ ]:
# Imports
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd

plt.rcParams['figure.dpi'] = 400
plt.rcParams['savefig.dpi'] = 400

## Global variables


In [ ]:
dirDir = "/home/users/l/lebrun/"
resultPath = dirDir + "results/"
dataPath = "/dpnc/beegfs/dampe/tryf/MC_Skimmed/"
filename = dataPath + "DmlNtup_Data_8years_FullPhotonData_AllSky.root"

## Load data : positions in degrees

In [ ]:
tree = None
f = ROOT.TFile(filename, "READ") 

tree = f.Get("DmlNtup")
print(tree)
print(type(tree))

df = ROOT.RDataFrame(tree)

In [ ]:
%%cpp 

double rad2deg(double x) {
    return x / (2*M_PI) * 360;
}

In [ ]:
df = df.Define("longitude_deg", "rad2deg(celestialcoordBest_2)").Define("latitude_deg", "rad2deg(celestialcoordBest_3)")

In [ ]:
# Test conversion

maxLat = df.Max("latitude_deg").GetValue()
minLat = df.Min("latitude_deg").GetValue()

maxLong = df.Max("longitude_deg").GetValue()
minLong = df.Min("longitude_deg").GetValue()

print(f"Latitude spans from {minLat} to {maxLat}")
print(f"Longitude spans from {minLong} to {maxLong}")

# Conclusion : conversion success

## Filter on Vela ROI

In [ ]:
dR = 1.5 # degrees, radius of region of interest
lon_Vela = 263.55
lat_Vela = -2.79 

lon_min = lon_Vela - dR
lon_max = lon_Vela + dR
lat_min = lat_Vela - dR
lat_max = lat_Vela + dR

dfVela = df.Filter(f"longitude_deg >= {lon_min}").Filter(f"longitude_deg <= {lon_max}").Filter(f"latitude_deg >= {lat_min}").Filter(f"latitude_deg <= {lat_max}")

## Plot ROI

### Create histogram

In [ ]:
# Create histogram 2D
h = dfVela.Histo2D(("VelaMap", "Events detected in Vela ROI;l [deg];b [deg]", 50, lon_min, lon_max, 50, lat_min, lat_max),
                        "longitude_deg", "latitude_deg")

### Plot colormap

In [ ]:
# Plot the map
c = ROOT.TCanvas()
h.Draw("MOLLWEIDE COLZ")
# Create and draw the marker
marker = ROOT.TMarker(lon_Vela, lat_Vela, 20)  # 20 is the marker style (solid circle)
marker.SetMarkerColor(ROOT.kRed)  # Set color to red
marker.SetMarkerSize(1.5)         # Set marker size
marker.Draw("same")
c.Update()
c.Draw()

### Temporary : save timestamps as csv

In [ ]:
timestamps = dfVela.AsNumpy(["tt_milisecond", "tt_second"])
print(timestamps)
timestamps = timestamps["tt_second"] + 1e-3*timestamps["tt_milisecond"]
np.savetxt(resultPath+"timestamps.csv", timestamps)

### Analyze timestamps

In [ ]:
print(f"The {timestamps.size} Timestamps run from {np.min(timestamps)} to {np.max(timestamps)} s")

fig2, ax2 = plt.subplots()
ax2.hist(timestamps, bins=200)
ax2.set_xlabel("timestamp [s]")
ax2.set_ylabel("counts")
fig2.show()

### Inspection of milisecond range

In [ ]:
timestamps.sort()

fig2, ax2 = plt.subplots()
ax2.hist(timestamps[:1000], bins=100)
ax2.set_xlabel("timestamp [s]")
ax2.set_ylabel("counts")
fig2.show()

### Conclusion : data inappropriate for period calculation to the ms 

## Try with known-period phase folding

In [ ]:
def phaseFold(timestamps, period):
    return (timestamps / period) % 1  # Convert times to phases

def twoPhaseFold(timestamps, period):
    return (timestamps / period) % 2  # Convert times to phases


In [ ]:
period_Vela = 0.089367  # Spolon et al 2019
phasesVela = twoPhaseFold(timestamps, period_Vela)
print(phasesVela.shape)
#phasesVela = np.concatenate((phasesVela, phasesVela + 1))

bin_edges = np.linspace(0, 2, num= 24+1)
print(bin_edges)

hist, bin_edges = np.histogram(phasesVela, bin_edges)
print(bin_edges)

figP, axP = plt.subplots()
axP.hist(phasesVela, bins=48, range=(0,2))
axP.set_ylim(30)
figP.show()

## Compute Rayleigh test

In [ ]:
# Define trial periods: scanning from 1 ms to 100 ms with fine resolution
P_min, P_max = 0.001, 0.1  # 1 ms to 100 ms
num_trials = 5000  # Resolution of the period search
trial_periods = np.linspace(P_min, P_max, num_trials)
# Compute the Rayleigh test for each trial period
Z1_values = []

trial_periods[-2] = 0.089

for P in trial_periods:
    phases = (timestamps / P) % 1  # Convert times to phases
    C = np.sum(np.cos(2 * np.pi * phases))
    S = np.sum(np.sin(2 * np.pi * phases))
    Z1 = (2 / len(timestamps)) * (C**2 + S**2)  # Rayleigh power
    Z1_values.append(Z1)
print(phases)
# Convert results to NumPy array for analysis
Z1_values = np.array(Z1_values)

# Identify the best candidate period (max Z1)
best_idx = np.argmax(Z1_values)
best_period = trial_periods[best_idx]

# Plot Rayleigh test results
fig3, ax3 = plt.subplots()
ax3.plot(trial_periods * 1000, Z1_values, label="Z₁² Statistic")  # Convert to ms
ax3.axvline(best_period * 1000, color='r', linestyle='--', label=f'Best P: {best_period*1000:.3f} ms')
ax3.set_xlabel("Trial Period (ms)")
ax3.set_ylabel("Z₁² Power")
ax3.set_title("Rayleigh Test Period Search")
ax3.legend()
ax3.grid()
fig3.show()

best_period